# 02 Creating and Querying Delta Tables

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Create Delta tables every way you'll meet in practice (SQL DDL, CTAS, <code>INSERT</code>, DataFrame writers, the <code>DeltaTable</code> builder), document them with comments and properties, inspect them (<code>DESCRIBE</code>, <code>SHOW CREATE TABLE</code>), and replace or drop them safely.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
The exam shows most table code in SQL: <code>CREATE TABLE</code>, <code>CREATE OR REPLACE TABLE ... AS SELECT</code>, <code>INSERT INTO</code>, <code>INSERT OVERWRITE</code>. Knowing which statement keeps history, which replaces the schema, and how to inspect a table you didn't create is everyday Databricks work.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Ways to create and fill a table</b><br>
| Statement / API | Creates table? | Keeps history? | Typical use |<br>|---|---|---|---|<br>| <code>CREATE TABLE t (cols ...)</code> | Yes, empty | n/a | Define a schema upfront (contracts, constraints) |<br>| <code>CREATE TABLE IF NOT EXISTS</code> | Only if missing | n/a | Idempotent setup scripts |<br>| <code>CREATE OR REPLACE TABLE t AS SELECT</code> (CTAS) | Yes, or replaces | <b>Yes</b>, a new version | Rebuild a table from a query |<br>| <code>DROP TABLE</code> + <code>CREATE TABLE</code> | Yes | <b>No</b>, history lost | Avoid: use <code>CREATE OR REPLACE</code> |<br>| <code>INSERT INTO t SELECT ...</code> | No | Yes | Append |<br>| <code>INSERT OVERWRITE t SELECT ...</code> | No | Yes | Replace all data, keep the schema |<br>| <code>df.write.saveAsTable</code> / <code>df.writeTo(...)</code> | Optional | Yes | DataFrame pipelines |<br>| <code>DeltaTable.createIfNotExists(spark)...</code> | Yes | n/a | Python builder with comments and properties |
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A team's setup script ran <code>DROP TABLE IF EXISTS customers; CREATE TABLE customers ...</code> on every deployment. After a bad deploy they wanted to roll back, but the history had been wiped by the <code>DROP</code>. Switching to <code>CREATE OR REPLACE TABLE</code> (or <code>CREATE TABLE IF NOT EXISTS</code> with <code>ALTER</code>s) keeps every version restorable.
</div>

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Sets the schema used by this folder.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The table prefix.
</div>

In [0]:
DB = "workspace.delta_lake"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")
print("tables in", DB)

## 1. CREATE TABLE with a schema, comments and properties

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Defines an empty table with column types, <code>NOT NULL</code>, column and table comments, and a table property.<br><br>
<b>Why it matters</b><br>An explicit DDL is the data contract: types, nullability and documentation live with the table, and Catalog Explorer and Genie show the comments to everyone.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>DESCRIBE TABLE EXTENDED</code> lists the columns with their comments, then table details including the comment and the <code>quality = silver</code> property.
</div>

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {DB}.customers (
        customer_id   BIGINT    NOT NULL COMMENT 'Business key from the CRM',
        name          STRING             COMMENT 'Full name',
        country       STRING             COMMENT 'ISO 3166-1 alpha-2 code',
        signup_date   DATE,
        lifetime_value DECIMAL(12,2)     COMMENT 'Total spend in USD'
    )
    USING DELTA
    COMMENT 'One row per customer (Silver layer)'
    TBLPROPERTIES ('quality' = 'silver')
""")
spark.sql(f"DESCRIBE TABLE EXTENDED {DB}.customers").show(30, truncate=False)

## 2. INSERT INTO and INSERT OVERWRITE

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Appends rows with <code>INSERT INTO ... VALUES</code> and <code>INSERT INTO ... SELECT</code>, then replaces all rows with <code>INSERT OVERWRITE</code>.<br><br>
<b>Why it matters</b><br><code>INSERT INTO</code> matches columns by <b>position</b> unless you list them, so list the columns. <code>INSERT OVERWRITE</code> replaces the data but keeps the table, its schema and its history.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
3 rows, then 4, then 2 after the overwrite. The history shows <code>WRITE</code> operations with mode <code>Append</code> and <code>Overwrite</code>.
</div>

In [0]:
spark.sql(f"""
    INSERT INTO {DB}.customers (customer_id, name, country, signup_date, lifetime_value) VALUES
      (1, 'Amal', 'IN', DATE'2023-01-15', 1200.50),
      (2, 'John', 'UK', DATE'2023-03-02', 560.00),
      (3, 'Vivek', 'AE', DATE'2024-07-21', 3100.00)
""")
print("after INSERT VALUES:", spark.table(f"{DB}.customers").count())

spark.sql(f"""
    INSERT INTO {DB}.customers (customer_id, name, country, signup_date, lifetime_value)
    SELECT 4, 'Sara', 'AE', DATE'2025-02-10', 75.00
""")
print("after INSERT SELECT:", spark.table(f"{DB}.customers").count())

spark.sql(f"""
    INSERT OVERWRITE {DB}.customers
    SELECT * FROM {DB}.customers WHERE lifetime_value > 1000
""")
print("after INSERT OVERWRITE:", spark.table(f"{DB}.customers").count())
spark.sql(f"DESCRIBE HISTORY {DB}.customers").select("version", "operation", "operationParameters.mode").show()

## 3. CTAS and CREATE OR REPLACE

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a table from a query (CTAS), then replaces it with a different query, and checks that the history survived.<br><br>
<b>Why it matters</b><br>CTAS infers the schema from the query, which is quick but less explicit than DDL. <code>CREATE OR REPLACE</code> is atomic and keeps the history. A <code>DROP</code> then <code>CREATE</code> loses it.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A <code>customers_by_country</code> table, then the replaced version with a new column, and a history with 2 versions.
</div>

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {DB}.customers_by_country
    COMMENT 'Customer counts per country'
    AS SELECT country, count(*) AS customers FROM {DB}.customers VERSION AS OF 2 GROUP BY country
""")
spark.table(f"{DB}.customers_by_country").show()

spark.sql(f"""
    CREATE OR REPLACE TABLE {DB}.customers_by_country AS
    SELECT country, count(*) AS customers, sum(lifetime_value) AS total_value
    FROM {DB}.customers VERSION AS OF 2 GROUP BY country
""")
spark.table(f"{DB}.customers_by_country").show()
spark.sql(f"DESCRIBE HISTORY {DB}.customers_by_country").select("version", "operation").show()

## 4. DataFrame writers and the DeltaTable builder

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates tables from Python: <code>saveAsTable</code>, <code>writeTo().createOrReplace()</code>, and the <code>DeltaTable</code> builder with column comments.<br><br>
<b>Why it matters</b><br>In PySpark pipelines you'll use the DataFrame APIs most. The builder is useful when you want a documented, empty table defined in Python, for example in a setup function.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Two tables created from DataFrames, then an empty <code>orders</code> table with a commented column, shown by <code>DESCRIBE TABLE</code>.
</div>

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

countries = spark.createDataFrame([("IN", "India"), ("UK", "United Kingdom"), ("AE", "UAE")], "code STRING, name STRING")
countries.write.mode("overwrite").saveAsTable(f"{DB}.countries")
countries.writeTo(f"{DB}.countries_v2").createOrReplace()
print("created:", [t.name for t in spark.catalog.listTables(DB) if t.name.startswith("countries")])

In [0]:
# The DeltaTable builder: an empty, documented table defined in Python
(
    DeltaTable.createOrReplace(spark)
    .tableName(f"{DB}.orders")
    .addColumn("order_id", "BIGINT", nullable=False)
    .addColumn("customer_id", "BIGINT", comment="References customers.customer_id")
    .addColumn("amount", "DECIMAL(10,2)")
    .addColumn("order_ts", "TIMESTAMP")
    .comment("Orders (Silver)")
    .execute()
)
spark.sql(f"DESCRIBE TABLE {DB}.orders").show(truncate=False)

## 5. Inspecting a table you didn't create

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Uses <code>SHOW TABLES</code>, <code>DESCRIBE DETAIL</code>, <code>SHOW CREATE TABLE</code> and <code>SHOW TBLPROPERTIES</code>.<br><br>
<b>Why it matters</b><br>These answer the first questions about any table: what exists, its format and size, how it was defined, and which properties (such as change data feed or retention) are set.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The list of tables in the schema, then details, the full <code>CREATE TABLE</code> statement for <code>customers</code>, and its properties.
</div>

In [0]:
spark.sql(f"SHOW TABLES IN {DB}").show(truncate=False)
spark.sql(f"DESCRIBE DETAIL {DB}.customers").select("name", "format", "numFiles", "sizeInBytes", "properties").show(truncate=False)
spark.sql(f"SHOW TBLPROPERTIES {DB}.customers").show(truncate=False)

In [0]:
# SHOW CREATE TABLE prints the full DDL (supported for Delta tables on Databricks)
print(spark.sql(f"SHOW CREATE TABLE {DB}.customers").first()[0])

## 6. Changing table metadata

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Adds a comment, changes a column comment, sets and unsets properties, and renames a table.<br><br>
<b>Why it matters</b><br>Documentation and properties evolve with the table. These are metadata-only operations: they create a new version in the history but don't rewrite data.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The new table comment and column comment in <code>DESCRIBE</code>, and the table available under its new name.
</div>

In [0]:
spark.sql(f"COMMENT ON TABLE {DB}.customers IS 'One row per active customer (Silver)'")
spark.sql(f"ALTER TABLE {DB}.customers ALTER COLUMN signup_date COMMENT 'First order date'")
spark.sql(f"ALTER TABLE {DB}.customers SET TBLPROPERTIES ('owner_team' = 'data-eng')")
spark.sql(f"ALTER TABLE {DB}.customers UNSET TBLPROPERTIES ('quality')")
spark.sql(f"ALTER TABLE {DB}.countries_v2 RENAME TO {DB}.countries_copy")

spark.sql(f"DESCRIBE TABLE {DB}.customers").filter("col_name = 'signup_date'").show(truncate=False)
print([t.name for t in spark.catalog.listTables(DB)])

## 7. Querying: by name, by version, and with the SQL features of Delta

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Queries a table by name, by version, filtered, and as a DataFrame.<br><br>
<b>Why it matters</b><br>A Delta table is queried like any table. Time travel adds <code>VERSION AS OF</code> / <code>TIMESTAMP AS OF</code>, and Databricks also supports the shorthand <code>table@v2</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Current customers, the version-1 snapshot, and the same snapshot read with <code>versionAsOf</code> in Python.
</div>

In [0]:
spark.sql(f"SELECT customer_id, name, country FROM {DB}.customers ORDER BY customer_id").show()
spark.sql(f"SELECT count(*) AS rows_v1 FROM {DB}.customers VERSION AS OF 1").show()
print("python versionAsOf 1:", spark.read.option("versionAsOf", 1).table(f"{DB}.customers").count())

## Under the hood

```
CREATE TABLE        -> commit 0: protocol + metaData (schema, comment, properties), no data files
INSERT / write      -> commit N: add data files
INSERT OVERWRITE    -> commit N: remove all current files + add new ones (old files kept until VACUUM)
CREATE OR REPLACE   -> commit N: new metaData + remove/add files, history continues
ALTER / COMMENT     -> commit N: new metaData only
DROP + CREATE       -> new table, new log: history is gone
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> DDL and DML run immediately when passed to `spark.sql`.

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Metadata only</span> comments, properties and renames change the log's metadata, not the data files.

In [0]:
spark.sql(f"DESCRIBE HISTORY {DB}.customers").select("version", "operation").show()

In [0]:
# Clean up the extra tables from this lesson (customers is reused later).
for t in ["customers_by_country", "countries", "countries_copy", "orders"]:
    spark.sql(f"DROP TABLE IF EXISTS {DB}.{t}")

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>TABLE_OR_VIEW_ALREADY_EXISTS</code></b><br>
Use <code>CREATE TABLE IF NOT EXISTS</code> for setup, or <code>CREATE OR REPLACE</code> to rebuild.<br><br>
<b>⛔ Problem: history lost after a redeploy</b><br><code>DROP TABLE</code> then <code>CREATE TABLE</code> starts a new table. Use <code>CREATE OR REPLACE</code>.<br><br>
**⛔ Problem: <code>INSERT INTO</code> puts values in the wrong columns**<br>It's positional. List the target columns explicitly.<br><br>
<b>⛔ Problem: CTAS creates unexpected types</b><br>CTAS infers types from the query. Cast in the <code>SELECT</code>, or define the table with DDL and <code>INSERT</code>.<br><br>
**⛔ Problem: <code>NOT NULL</code> column rejects an insert**<br>Delta enforces <code>NOT NULL</code>. Provide a value, or remove the constraint deliberately.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is the difference between <code>CREATE OR REPLACE TABLE</code> and <code>DROP TABLE</code> followed by <code>CREATE TABLE</code>?</b><br>
<code>CREATE OR REPLACE</code> is a single atomic operation that writes a new version, so the history and time travel survive, and readers never see a missing table. <code>DROP</code> plus <code>CREATE</code> creates a brand-new table with no history, and there's a moment when the table doesn't exist.<br><br>

<b>🎤 2. What is the difference between <code>INSERT INTO</code> and <code>INSERT OVERWRITE</code>?</b><br>
<code>INSERT INTO</code> appends rows. <code>INSERT OVERWRITE</code> replaces all existing data (or the matching partitions) in a new version, keeping the table definition and history.<br><br>

<b>🎤 3. What is CTAS, and what are its trade-offs?</b><br>
<code>CREATE TABLE ... AS SELECT</code> creates and fills a table from a query in one step. It's convenient, but the schema is inferred from the query, so you don't declare constraints or exact types upfront unless you cast.<br><br>

<b>🎤 4. How do you find out how a table was defined?</b><br>
<code>SHOW CREATE TABLE</code>, <code>DESCRIBE TABLE EXTENDED</code> for columns, comments and details, <code>DESCRIBE DETAIL</code> for format, files, size and properties, and <code>DESCRIBE HISTORY</code> for how it changed.<br><br>

<b>🎤 5. Why add comments to tables and columns?</b><br>
They document meaning, units and ownership where everyone sees them: Catalog Explorer, lineage, and AI tools such as Genie use them to understand the data.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. Which statement creates a table from a query and replaces it if it already exists, while preserving the table's history?</b><br>
A. <code>DROP TABLE t; CREATE TABLE t AS SELECT ...</code><br>
B. <code>CREATE OR REPLACE TABLE t AS SELECT ...</code><br>
C. <code>INSERT INTO t SELECT ...</code><br>
D. <code>CREATE TABLE IF NOT EXISTS t AS SELECT ...</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> D does nothing if the table exists. A loses the history.</details><br><br>

<b>Q2. A data engineer wants to replace all rows of an existing table with the result of a query, without changing the table's schema or properties. Which statement fits best?</b><br>
A. <code>INSERT OVERWRITE t SELECT ...</code><br>
B. <code>MERGE INTO t ...</code><br>
C. <code>CREATE TABLE t LIKE src</code><br>
D. <code>ALTER TABLE t REPLACE</code><br>
<details><summary><b>Show answer</b></summary><b>A.</b></details><br><br>

<b>Q3. Which command shows the format, number of files and size of a Delta table?</b><br>
A. <code>DESCRIBE HISTORY</code><br>
B. <code>DESCRIBE DETAIL</code><br>
C. <code>SHOW TABLES</code><br>
D. <code>SHOW PARTITIONS</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Create a <code>stores</code> table with DDL: <code>store_id INT NOT NULL</code>, <code>city STRING</code>, <code>opened DATE</code>, with a table comment and a property <code>domain = retail</code></li><li>Insert 3 stores, listing the columns explicitly</li><li>Replace the data with only the stores opened after 2020 using <code>INSERT OVERWRITE</code></li><li>Rebuild it with <code>CREATE OR REPLACE TABLE ... AS SELECT</code> adding a column <code>years_open</code>, and show that the history has all versions</li><li>Show the <code>CREATE TABLE</code> statement and drop the table</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
stores = f"{DB}.stores"

# 1. DDL with comment and property
spark.sql(f"""
    CREATE OR REPLACE TABLE {stores} (store_id INT NOT NULL, city STRING, opened DATE)
    COMMENT 'Physical stores' TBLPROPERTIES ('domain' = 'retail')
""")

# 2. Explicit column list
spark.sql(f"""
    INSERT INTO {stores} (store_id, city, opened) VALUES
    (1, 'Chennai', DATE'2015-04-01'), (2, 'Dubai', DATE'2021-09-15'), (3, 'London', DATE'2023-01-10')
""")

# 3. Overwrite with a filtered copy
spark.sql(f"INSERT OVERWRITE {stores} SELECT * FROM {stores} WHERE opened > DATE'2020-12-31'")

# 4. Rebuild with an extra column, history preserved
spark.sql(f"""
    CREATE OR REPLACE TABLE {stores} AS
    SELECT *, floor(datediff(current_date(), opened) / 365.25) AS years_open FROM {stores}
""")
spark.sql(f"DESCRIBE HISTORY {stores}").select("version", "operation").show()

# 5. Definition (Databricks), then drop
try:
    print(spark.sql(f"SHOW CREATE TABLE {stores}").first()[0])
except Exception as e:
    print("SHOW CREATE TABLE not available here:", type(e).__name__)
spark.sql(f"DROP TABLE {stores}")

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>DDL (<code>CREATE TABLE</code>) for explicit contracts. CTAS for quick tables from queries</li><li><code>CREATE OR REPLACE</code> keeps history. <code>DROP</code> + <code>CREATE</code> loses it</li><li><code>INSERT INTO</code> appends (list the columns). <code>INSERT OVERWRITE</code> replaces data and keeps the definition</li><li>Python: <code>saveAsTable</code>, <code>writeTo().createOrReplace()</code>, <code>DeltaTable.createOrReplace(spark)...</code></li><li>Inspect: <code>SHOW TABLES</code>, <code>DESCRIBE [DETAIL | HISTORY | TABLE EXTENDED]</code>, <code>SHOW CREATE TABLE</code>, <code>SHOW TBLPROPERTIES</code></li><li>Document with <code>COMMENT</code> and table properties</li></ul>
</div>

| Task | SQL |
|---|---|
| Create | `CREATE TABLE IF NOT EXISTS t (c INT NOT NULL COMMENT '...') COMMENT '...'` |
| Create from query | `CREATE OR REPLACE TABLE t AS SELECT ...` |
| Append | `INSERT INTO t (a, b) SELECT ...` |
| Replace data | `INSERT OVERWRITE t SELECT ...` |
| Inspect | `DESCRIBE DETAIL t`, `SHOW CREATE TABLE t` |
| Document | `COMMENT ON TABLE t IS '...'`, `ALTER TABLE t ALTER COLUMN c COMMENT '...'` |
| Properties | `ALTER TABLE t SET TBLPROPERTIES ('k' = 'v')` |

## Git commit

```
git add 03_delta_lake/02_creating_and_querying_tables.ipynb
git commit -m "add 03_02 creating and querying tables notebook"
```